Multi-Static Data Fusion Performance Evaluation and Running Example
===================================================================

This example demonstrates the use of the multistatic tracking algorithm developed by University of Liverpool
that has been integrated into Eurybia’s Multistatic Data Fusion Engine (MSDFE). It also constitudes part of
deliverable D3 of the Multi-Static Tracking project.

This notebook includes the following:

1. Example of how to configure and run the Local Trackers.
2. Example of how to configure and run the Fusion Tracker.
3. Example of how to evaluate the performance of the Local Trackers and Fusion Tracker.

Importing required libraries
----------------------------

In [ ]:
%matplotlib widget
import datetime
from copy import deepcopy
from enum import Enum
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from stonesoup.dataassociator.mfa import MFADataAssociator
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment
from stonesoup.deleter.error import MeasurementCovarianceBasedDeleter
from stonesoup.deleter.multi import CompositeDeleter
from stonesoup.deleter.time import UpdateTimeStepsDeleter
from stonesoup.gater.distance import DistanceGater
from stonesoup.hypothesiser.mfa import MFAHypothesiser
from stonesoup.hypothesiser.probability import PDAHypothesiser, PDAHypothesiserNoPrediction
from stonesoup.initiator.simple import MultiMeasurementInitiatorMixture
from stonesoup.initiator.twostate import TwoStateMeasurementInitiatorMixture
from stonesoup.measures import Mahalanobis
from stonesoup.metricgenerator.metrictables import SIAPTableGenerator
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, OrnsteinUhlenbeck, \
    NthDerivativeDecay
from stonesoup.predictor.kalman import  UnscentedKalmanPredictor
from stonesoup.predictor.twostate import TwoStatePredictor
from stonesoup.reader.niag import STANAGContactReader
from stonesoup.reader.track import TrackReader
from stonesoup.reader.tracklet import TrackletExtractor, PseudoMeasExtractor
from stonesoup.tracker.fuse import FuseTracker
from stonesoup.tracker.simple import MultiTargetMultiMixtureTracker
from stonesoup.types.array import StateVector, CovarianceMatrix
from stonesoup.types.numeric import Probability
from stonesoup.types.state import GaussianState
from stonesoup.types.update import Update
from stonesoup.updater.kalman import UnscentedKalmanUpdater
from stonesoup.updater.twostate import TwoStateKalmanUpdater

from stonesoup.functions.plotting_utils import plot_gnd, plot_platform, plot_gospa
from stonesoup.functions.metric_utils import gen_metric_fuse, prepare_tracks_fuse

Datasets
--------
In this section we define information pertinent to the provided datasets

**NOTE: The paths to the STANAG messages directories should be updated to the correct paths on your system.**

In [ ]:
plot_coord = 'xyz'                        # Coordinate system to plot in
ref_lat=49.725                            # Reference latitude for coordinate conversion
ref_lon=-4.85                             # Reference longitude for coordinate conversion
stanag_config = 'NIAGSparse'              # Configuration subfolder for STANAG messages
stanag_meta_header_name = 'LatencyHeader' # Meta header name for STANAG messages
rx_plat_id_selects = [1, 2]
class Dataset(Enum):
    """Enumeration of the datasets to use"""
    SIM1 = 'Sim1'    # Simulated dataset 1 - Single target
    SIM2 = 'Sim2'    # Simulated dataset 2 - Multiple targets
    REAL = 'Real'    # Real dataset

# Path to the STANAG messages directory for each dataset
# IMPORTANT: Update the paths to the correct paths on your system
stanag_msg_directories = {
    Dataset.SIM1: Path(r'C:\Users\sglvladi\OneDrive\Documents\University of Liverpool\PostDoc\EURYBIA - Dstl\Data\Drop 2 - 13Feb2025\20250213_UoLExample'),
    Dataset.SIM2: Path(r'C:\Users\sglvladi\OneDrive\Documents\University of Liverpool\PostDoc\EURYBIA - Dstl\Data\Drop 3 - 05Mar2025\20250305_UoL_Sim_Two_O'),
    Dataset.REAL: Path(r'C:\Users\sglvladi\OneDrive\Documents\University of Liverpool\PostDoc\EURYBIA - Dstl\Data\Drop 4 - 11Mar2025\20250305_UoL_Real_Three_OS')
}

**Below we define which dataset we will be running**

In [ ]:
# Define the dataset to use. Possible values [Dataset.SIM1, Dataset.SIM2, Dataset.REAL]
DATASET = Dataset.SIM1

Parameters
----------
In this section, we define the various parameters required to run the example against the different datasets.

### Common Parameters

We begin by defining the parameters that are common to all datasets. These include the following:

In [ ]:
q = 0.01                                 # Process noise (q)
q_bias_range = 1e-1                      # Bias process noise for range (q_b^r)
q_bias_bearing = np.radians(1e-7)        # Bias process noise for bearing (q_b^b)
decay_factor = 0.0001                    # Decay factor (K)
sigma_r = 200.                           # Range measurement noise (sigma_r)
time_steps_since_update = 5              # Time steps since last update to delete track (M_{del})
max_bearing_std = np.radians(15)         # Maximum bearing standard deviation for track deletion (sigma_{b}^{max})
slide_window = 2                         # Sliding window size for MFA

# Prior for the local trackers
#The order of the state vector is [x, vx, y, vy, b, r], where b is the bearing bias and r is the range bias.
local_prior = GaussianState(StateVector([0., 0., 0., 0., 0., 0.]),
                            CovarianceMatrix(np.diag([0, 10., 0, 10., np.pi / 6, 50.]) ** 2))
# Prior for the fusion tracker
# The order of the state vector is [x, vx, y, vy], where x and y sadare the position, and vx and vy are the velocity.
fuse_prior = GaussianState(StateVector([0., 0., 0., 0.]),
                           CovarianceMatrix(np.diag([0., 10., 0., 10.]) ** 2))

### Dataset Specific Parameters

The parameters below are specific to each dataset. These include the following:

In [ ]:

if DATASET == Dataset.SIM1:
    sigma_b = np.radians(.2)                            # Bearing measurement noise (sigma_b)
    fuse_interval = datetime.timedelta(minutes=10)      # Fusion interval (\Delta t_f)
    prob_detect = 1.                                    # Detection probability (P_D)
    clutter_rate = 0.0001                               # Mean number of clutter points per scan (lambda_{FA})
    max_range = 10000                                   # Max range of sensor (r_{max})
    surveillance_area = np.pi*max_range**2              # Surveillance region area
    clutter_density = clutter_rate/surveillance_area    # Mean number of clutter points per unit area (lambda)
    init_threshold = 2                                  # Initial threshold for the initiator (M_{init})
    max_range_std = 2e3                                 # Maximum range standard deviation for track deletion (sigma_{r}^{max})
    init_clutter_density = clutter_density              # Initial clutter density
    xlim = [0, 25000]                                   # X-axis limits for the plot
    ylim = [0, 75000]                                   # Y-axis limits for the plot
    snr_threshold = 10                                  # SNR threshold for the contacts reader
    update_rate = None                                  # Update rate for the contacts reader (default is None)
    target_plat_unit_id = [(3, 1)]                      # Target platform unit ID (used to extract ground truth from contacts reader)
elif DATASET == Dataset.SIM2:
    sigma_b = np.radians(2.)                            # Bearing measurement noise (sigma_b)
    fuse_interval = datetime.timedelta(minutes=2)       # Fusion interval (\Delta t_f)
    prob_detect = 0.9                                   # Detection probability (P_D)
    clutter_rate = 10                                   # Mean number of clutter points per scan (lambda_{FA})
    max_range = 35000                                   # Max range of sensor (r_{max})
    surveillance_area = np.pi * max_range ** 2          # Surveillance region area
    clutter_density = clutter_rate / surveillance_area  # Mean number of clutter points per unit area (lambda)
    init_threshold = 10                                 # Initial threshold for the initiator (M_{init})
    max_range_std = 1e3                                 # Maximum range standard deviation for track deletion (sigma_{r}^{max})
    init_clutter_density = 1e-3                         # Initial clutter density
    xlim = [-15000, 15000]                              # X-axis limits for the plot
    ylim = [-15000, 15000]                              # Y-axis limits for the plot
    snr_threshold = 10                                  # SNR threshold for the contacts reader
    update_rate = None                                  # Update rate for the contacts reader (default is None)
    target_plat_unit_id = [(3,1), (4,1), (91,1), (92,1), (93,1)] # Target platform unit ID (used to extract ground truth from contacts reader)
elif DATASET == Dataset.REAL:
    sigma_b = np.radians(2.)                            # Bearing measurement noise (sigma_b)
    fuse_interval = datetime.timedelta(seconds=40)      # Fusion interval (\Delta t_f)
    prob_detect = 0.9                                   # Detection probability (P_D)
    clutter_rate = 7                                    # Mean number of clutter points per scan (lambda_{FA})
    max_range = 15000                                   # Max range of sensor (r_{max})
    surveillance_area = np.pi * max_range ** 2          # Surveillance region area
    clutter_density = clutter_rate / surveillance_area  # Mean number of clutter points per unit area (lambda)
    init_threshold = 10                                 # Initial threshold for the initiator (M_{init})
    max_range_std = 1e3                                 # Maximum range standard deviation for track deletion (sigma_{r}^{max})
    init_clutter_density = 1e-3                         # Initial clutter density
    xlim = [-15000, 15000]                              # X-axis limits for the plot
    ylim = [-15000, 15000]                              # Y-axis limits for the plot
    snr_threshold = 14                                  # SNR threshold for the contacts reader
    update_rate = datetime.timedelta(seconds=20)        # Update rate for the contacts reader (set to 20 seconds to ensure scans are not missed)
    target_plat_unit_id = [(4, 1)]                      # Target platform unit ID (used to extract ground truth from contacts reader)

Local Tracker Configuration
---------------------------
In this section, we configure the local trackers. The local trackers are configured using the following components:

### Transition Model

We begin by defining the transition model for the local trackers. The transition model is a combined linear Gaussian
transition model that consists of two Ornstein-Uhlenbeck processes for the Cartesian coordinates and two NthDerivativeDecay
processes for the biases.

In [ ]:
local_transition_model = CombinedLinearGaussianTransitionModel([OrnsteinUhlenbeck(q, decay_factor),
                                                               OrnsteinUhlenbeck(q, decay_factor),
                                                               NthDerivativeDecay(0, q_bias_bearing, decay_factor),
                                                               NthDerivativeDecay(0, q_bias_range, decay_factor)])

### Predictor and Updater

We then define the predictor and updater components for the local trackers. The Unscented Kalman Filter (UKF) is used
for both the predictor and updater components.

In [ ]:
predictor = UnscentedKalmanPredictor(local_transition_model)
updater = UnscentedKalmanUpdater(None, True)

### Hypothesiser and Data Associator

We define the hypothesiser and data associator components for the local trackers. The Multi-Frame Assignment (MFA)
algorithm is used for both the hypothesiser and data associator components. Internally, the MFA Hypothesiser uses
the PDA Hypothesiser for the hypothesis generation and likelihood calculation. The PDA Hypothesiser is wrapped
with a Distance Gater to gate out detections that are not within a certain distance from the predicted state.

In [ ]:
hypothesiser = PDAHypothesiser(predictor, updater, clutter_density, prob_detect)
hypothesiser = DistanceGater(hypothesiser, Mahalanobis(), 10)
hypothesiser = MFAHypothesiser(hypothesiser)
data_associator = MFADataAssociator(hypothesiser, slide_window=slide_window)

### Deleter
We define the deleter component for the local trackers. The deleter component is a composite deleter that consists
of two sub-deleters: one that deletes tracks based on the time steps since the last update and another that deletes
tracks based on the measurement covariance matrix values.

In [ ]:
deleter1 = UpdateTimeStepsDeleter(time_steps_since_update)
deleter2 = MeasurementCovarianceBasedDeleter([max_bearing_std ** 2, max_range_std ** 2])
deleter = CompositeDeleter([deleter1, deleter2], intersect=False)

### Initiator
We define the initiator component for the local trackers. The initiator component is a Multi-Measurement Initiator
that runs an internal tracker to maintain tentative tracks and initiates new tracks when said tentative tracks are
successfully associated with detections for a certain number of time steps (M_{init}).

The initiator uses a Global Nearest Neighbour (GNN) data associator to associate detections with tentative tracks,
while, similar to above, the PDA Hypothesiser is used for hypothesis generation and likelihood calculation. A slightly
higher clutter density is used for the initiator to avoid prematurely initiating tracks.

The initiator also uses the same predictor, updater and deleter components as the local trackers.

In [ ]:
hypothesiser_init = PDAHypothesiser(predictor, updater, init_clutter_density, prob_detect)
hypothesiser_init = DistanceGater(hypothesiser_init, Mahalanobis(), 10)
data_associator_init = GNNWith2DAssignment(hypothesiser_init)
initiator = MultiMeasurementInitiatorMixture(local_prior, None, deleter, data_associator_init, updater, init_threshold)

### Contact Readers

We then configure the contact readers for the local trackers. The contact readers are used to read the STANAG messages
and extract the contac

In [ ]:
readers = []
stanag_msg_directory = stanag_msg_directories[DATASET]  # Path to the STANAG messages directory
for i, rx_plat_id_select in enumerate(rx_plat_id_selects):
    # Detector/Reader
    contact_reader = STANAGContactReader(stanag_msg_directory,
                                          state_vector_fields=("RelBearing", "RX2contact_range"),
                                          time_field = None,
                                          snr_threshold=snr_threshold,
                                          rerr=sigma_r**2,
                                          berr=sigma_b**2,
                                          endianness = 0,
                                          stanag_msg_directory=stanag_msg_directory,
                                          reference_lat = ref_lat,
                                          reference_lon = ref_lon,
                                          with_bias=True,
                                          update_rate=update_rate)
    contact_reader.read_stanag_files(rx_plat_id_select=rx_plat_id_select,
                                      config_subfolder=stanag_config,
                                      meta_header_name=stanag_meta_header_name)
    contact_reader.get_stanag_ground_truth_from_SM01(target_plat_unit_id=target_plat_unit_id)
    readers.append(contact_reader)

### Local Trackers

Finally we configure the local trackers using the components defined above. Since the Multi Measurement Initiator is not
memoryless --- it maintains tentative tracks, pertinent to each tracker --- we need to create a new instance of the
initiator for each sensor. We do this by creating a deep copy of the initiator for each sensor.

We also wrap each local tracker with a TrackReader, which is responsible for feeding the tracks to the Fusion Tracker.
This is also particularly useful when running the trackers with a stream of real data, where the tracks are generated
asynchronously and need to be fed to the Fusion Tracker as they are generated. This is done by setting the `run_async`
parameter to `True` in the TrackReader. In this example, we set `run_async=False` to run the trackers synchronously,
since we know that both sensors have matching update rates.

Also note that the iteration over the readers is done using the `enumerate` function to generate a ``sensor_id``,
which is then fed to the `TrackReader` to distinguish between the different sensors. It is important to assign a
unique sensor ID to each sensor, as this is used by the Fusion Tracker to distinguish between the different sensors.
Also, sensor ids should be integers, and the ``local_trackers`` list should be sorted by sensor ID.
This is important for the correct operation of the Fusion Tracker. Here, the sorting is performed implicitly
by the `enumerate` function.

In [ ]:
local_trackers = []
for sensor_id, contact_reader in enumerate(readers):
    # Create a deep copy of the initiator for each sensor
    initiator_tmp = deepcopy(initiator)
    # Initialise Local Tracker
    local_tracker = MultiTargetMultiMixtureTracker(initiator_tmp, deleter, contact_reader, data_associator, updater)
    # Wrap the local tracker with a TrackReader and append to the local trackers list
    local_trackers.append(
        TrackReader(local_tracker, run_async=False, transition_model=local_transition_model, sensor_id=sensor_id)
    )

Fusion Tracker Configuration
----------------------------
In this section, we configure the Fusion Tracker. The Fusion Tracker is configured using the following components:

### Transition model

We begin by defining the transition model for the Fusion Tracker. The transition model is a combined linear Gaussian
transition model that consists of two Ornstein-Uhlenbeck processes for the Cartesian coordinates.

In [ ]:
transition_model = CombinedLinearGaussianTransitionModel([OrnsteinUhlenbeck(q, decay_factor),
                                                          OrnsteinUhlenbeck(q, decay_factor)])

### Tracklet Extractor

We then define the tracklet extractor component for the Fusion Tracker. The tracklet extractor is responsible for
extracting tracklets from the local trackers. The tracklet extractor is configured with the local trackers, the
transition model, and the fusion interval.

Note that as a sanity check, we ensure the local trackers are sorted by sensor ID before passing them to the
tracklet extractor.

In [ ]:
local_trackers = sorted(local_trackers, key=lambda x: x.sensor_id)
tracklet_extractor = TrackletExtractor(trackers=local_trackers,
                                       transition_model=transition_model,
                                       fuse_interval=fuse_interval)

### Pseudo Measurement Extractor

We then define the Pseudo Measurement Extractor component for the Fusion Tracker. The Pseudo Measurement Extractor is
responsible for extracting measurements from the tracklets generated by the tracklet extractor. The Pseudo Measurement
Extractor then becomes the detector component for the Fusion Tracker.

Along with the tracklet extractor, the Pseudo Measurement Extractor is configured with the state indices to use from
the tracklet states. In this example, we use the first four state indices, which correspond to the Cartesian
position and velocity coordinated.

In [ ]:
detector = PseudoMeasExtractor(tracklet_extractor,
                               state_idx_to_use=[0,1,2,3])

### Predictor and Updater

We proceed to define the predictor and updater components for the Fusion Tracker. The Two State Predictor and Updater
are used for the predictor and updater components, respectively.

In [ ]:
two_state_predictor = TwoStatePredictor(transition_model)
two_state_updater = TwoStateKalmanUpdater(None, True)

### Hypothesiser and Data Associator

Next, we define the hypothesiser and data associator components for the Fusion Tracker. The Multi-Frame Assignment (MFA)
algorithm is used for the hypothesiser and data associator components. Similar to the local trackers, the MFA Hypothesiser
uses the PDA Hypothesiser for the hypothesis generation and likelihood calculation.

In [ ]:
fuse_hypothesiser = PDAHypothesiserNoPrediction(predictor=None,
                                                updater=two_state_updater,
                                                clutter_spatial_density=Probability(-80, log_value=True),
                                                prob_detect=Probability(.9),
                                                prob_gate=Probability(0.99))
fuse_hypothesiser = DistanceGater(fuse_hypothesiser, Mahalanobis(), 10)
fuse_hypothesiser = MFAHypothesiser(fuse_hypothesiser)
fuse_associator = MFADataAssociator(fuse_hypothesiser, slide_window=slide_window) # in Fuse tracker

### Initiator

The final component to configure for the Fusion Tracker is the initiator. The initiator component for the Fusion Tracker
is a TwoStateMeasurementInitiatorMixture that uses the prior and transition model defined earlier, along with the
TwoStateUpdater to initialise new tracks for all unassociated detections.

In [ ]:
fuse_initiator = TwoStateMeasurementInitiatorMixture(fuse_prior, transition_model, two_state_updater)

### Fusion Tracker

Finally, we configure the Fusion Tracker using the components defined above. Along with the components, the Fusion Tracker
is configured with the `death_rate` parameter, which is used to define the decline in the probability of existence of
a track, per second, when no detections are associated with the track. The `delete_thresh` parameter is used to define
the threshold for deleting tracks based on the probability of existence. Finally, the `prob_detect` parameter is used to
define the detection probability for the Fusion Tracker.

In [ ]:
fuse_tracker = FuseTracker(initiator=fuse_initiator, predictor=two_state_predictor,
                           updater=two_state_updater, associator=fuse_associator,
                           detector=detector, death_rate=1e-4,
                           prob_detect=Probability(.9),
                           delete_thresh=Probability(0.1))

Estimation
----------
In this section, we run the local trackers and the Fusion Tracker to estimate the tracks. Iterating over the local
trackers is performed implicitly by iterating over the `fuse_tracker`, which in triggers the Stone Soup framework to
iterate over the local trackers.

We also keep a record of the quantities of interest, such as the tracks, detections, and ground truth, to use later
for plotting and evaluation.

In [ ]:
timestamps = []                                                         # List to store the timestamps
all_fused_tracks = set()                                                # Set to store all fused tracks
all_detections = {i: set() for i in range(len(readers))}                # Dictionary to store all detections for each sensor
local_tracks = {i: set() for i in range(len(local_trackers))}           # Dictionary to store all local tracks for each sensor
for time, ctracks in fuse_tracker:
    print(f'Time: {time} | Number of tracks: {len(ctracks)}')
    # Update the timestamps
    timestamps.append(time)
    # Update the fused tracks
    all_fused_tracks.update(ctracks)
    # Update the local tracks for each sensor
    for i, tracker in enumerate(local_trackers):
        local_tracks[i].update(tracker.current[1])
    # Update the detections for each sensor
    for i, reader in enumerate(readers):
        all_detections[i].update(reader.detections)

Evaluation
----------
In this section, we evaluate the performance of the local trackers and the Fusion Tracker. We begin by plotting the
ground truth, detections, tracklets, and tracks. We then evaluate the performance of the local trackers and the Fusion
Tracker using the Generalised Optimal Subpattern Assignment (GOSPA) and Single Integrated Air Picture (SIAP) metrics.

### Plotting

We begin by plotting the ground truth, detections, tracklets, and tracks.

In [ ]:
# Extract the ground truths from the contact readers
ground_truths = {track for track in readers[0].ground_truth.values()}

# Create a figure and axis for plotting
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1)
plt.ion()
ax.set_xlim(xlim)
ax.set_ylim(ylim)
colors = ['r', 'g', 'b']    # Colors for the different sensors
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.cla()
ax.set_xlabel('East')
ax.set_ylabel('North')
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# Plot the sensor platforms
ax.plot([], [], 'bo', label='TX')
ax.plot([], [], 'mo', label='RX')
for reader in readers:
    plot_platform(reader.truth_TX, ref_lat, ref_lon, ax, plot_coord, 'b', )
    plot_platform(reader.truth_RX, ref_lat, ref_lon, ax, plot_coord, 'm', )

# Plot the ground truths
plot_gnd(ground_truths, ref_lat, ref_lon, ax, plot_coord)
# Plot the detections
# for i, (detections, color) in enumerate(zip(all_detections.values(), colors)):
#     # Assign sensor label for convenience
#     sensor_label = 'TX/RX' if i == 0 else 'RX'
#     ax.plot([], [], f'{color}x', label=f'{sensor_label} Sensor Detections')
#     for detection in detections:
#         x, y = detection.measurement_model.inverse_function(detection)[[0, 2]]
#         ax.plot(x, y, f'{color}x')
# Plot the local tracks
for i, (tracks, color) in enumerate(zip(local_tracks.values(), colors)):
    # Assign sensor label for convenience
    sensor_label = 'TX/RX' if i == 0 else 'RX'
    ax.plot([], [], f':.{color}', label=f'{sensor_label} Sensor Tracks')
    for track in tracks:
        data = np.array([s.mean for s in track.states if isinstance(s, Update)])
        idx = [0, 2]
        plt.plot(data[:, idx[0]], data[:, idx[1]], f':.{color}')
# Plot the fused tracks
plt.plot([], [], '-*c', label='Fused Tracks')
for track in all_fused_tracks:
    data = np.array([state.state_vector for state in track])
    plt.plot(data[:, 4], data[:, 6], '-*c')
ax.legend()

We then plot the bearing and range biases for each local tracker.

In [ ]:
for i, tracks in local_tracks.items():
    fig2 = plt.figure(figsize=(10, 4))
    ax2, ax3 = fig2.subplots(1, 2)
    ax2.set_title('Bearing Bias')
    ax2.set_ylabel('Bearing (deg)')
    ax3.set_title('Range Bias')
    ax3.set_ylabel('Range (m)')
    ax3.set_xlabel('Track Timestep')
    ax2.set_xlabel('Track Timestep')
    for track in tracks:
        data = np.array([state.state_vector for state in track.states])
        num_steps = len(data)
        b_bias = np.degrees(data[:, -2].ravel())
        ax2.plot([i for i in range(num_steps)], b_bias, 'r-')
        sd = np.degrees(np.sqrt(np.squeeze([state.covar[-2, -2] for state in track.states])))
        ax2.fill_between([i for i in range(num_steps)], b_bias - sd, b_bias + sd, facecolor='g', alpha=0.5)
        ax3.plot([i for i in range(num_steps)], data[:, -1], 'r-')
        sd = np.sqrt(np.squeeze([state.covar[-1, -1] for state in track.states]))
        ax3.fill_between([i for i in range(num_steps)], data[:, -1].ravel() - sd, data[:, -1].ravel() + sd, facecolor='g', alpha=0.5)

### Performance Evaluation

Finally, we calculate and plot the GOSPA and SIAP metrics for the local trackers and the Fusion Tracker.
We begin by preparing the tracks for evaluation by fusing the local tracks and the fused tracks. The preparation step
uses a TrackToTruthAssociator to associate the tracks with the ground truth for Datasets SIM1 and REAL.

In [ ]:
metric_key_to_label = {
    'fuse': 'Fusion Engine',
    'local_0': 'Local Tracker - TX/RX Sensor',
    'local_1': 'Local Tracker - RX Sensor',
}
fused_tracks, local_tracks = prepare_tracks_fuse(DATASET, timestamps, ground_truths, all_fused_tracks, local_tracks)

Next, we walculate the GOSPA and SIAP metrics for the local trackers and the Fusion Tracker

In [ ]:
gospa_metrics = gen_metric_fuse('GOSPA', timestamps, ground_truths, fused_tracks, local_tracks)
siap_metrics = gen_metric_fuse('SIAP', timestamps, ground_truths, fused_tracks, local_tracks)

We plot the SIAP metrics for the local trackers and the Fusion Tracker

In [ ]:
for key, siap_metric in siap_metrics.items():
    siap_averages = {metric for metric in siap_metric
                     if metric.title.startswith("SIAP") and not metric.title.endswith(" at times")}
    siap_time_based = {metric for metric in siap_metric if metric.title.endswith(' at times')}
    _ = SIAPTableGenerator(siap_averages).compute_metric()
    plt.title(f'SIAP Averages\n{metric_key_to_label[key]}')

Finally, we plot the GOSPA metrics for the local trackers and the Fusion Tracker

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
for key, metric in gospa_metrics.items():
    plot_gospa(metric, f'{metric_key_to_label[key]}', ax=ax)
ax.set_ylabel("GOSPA distance")
ax.tick_params(labelbottom=False)
_ = ax.set_xlabel("Time")
ax.set_title(f'GOSPA metrics')
plt.tight_layout()
plt.legend()